# Optuna pytorch

Original experiment from [Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow](https://github.com/Muhammad-Huzifa/Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow/blob/dfa17b2b298631b10e5904b2f1c39ad64d424bc1/Pytorch/Hyperparameter%20Tuning%20using%20Optuna.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.optim as  optim
import torch.nn as nn

Download and extract the dataset described in `docs/DATASETS.md` before running the model cells. Notebook-specific credential uploads and mounted-drive setup have been removed.

In [ ]:
! pip install opendatasets

Download and extract the dataset described in `docs/DATASETS.md` before running the model cells. Notebook-specific credential uploads and mounted-drive setup have been removed.

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv(str(TABULAR / 'fashion-mnist_train.csv'))

In [ ]:
data.head()


In [ ]:
torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
torch.manual_seed(42)

In [ ]:
fig , axes  = plt.subplots(4,4,figsize = (10,10))
fig.suptitle("First 16 images"  ,fontsize = 16)

for i , ax in enumerate(axes.flat):
  img = data.iloc[i , 1:].values.reshape(28,28)
  ax.imshow(img)
  ax.axis('off')
  ax.set_title(f"Label {data.iloc[i,0]}")
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

In [ ]:
X = data.iloc[: , 1:].values
y = data.iloc[: , 0].values



In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [ ]:
X_train = X_train / 255
X_test = X_test/255

In [ ]:
class CustomDataset(Dataset):
  def __init__(self ,features, labels):
    self.features = torch.tensor(features , dtype = torch.float32)
    self.labels = torch.tensor(labels , dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self , idx):
    return self.features[idx] , self.labels[idx]

In [ ]:


train_dataset  = CustomDataset(X_train , y_train)
test_dataset = CustomDataset(X_test , y_test)

In [ ]:
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle = True ,pin_memory=True)
test_loader = DataLoader(test_dataset , batch_size = 32 , shuffle = False , pin_memory=True)

In [ ]:
class MyNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self, x):

    return self.model(x)

**Objective Function**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

def objective(trial):
    # Suggest hyperparameters
    hidden_layer = trial.suggest_int("hidden_layer", 1, 5)
    num_neurons = trial.suggest_int("num_neurons", 8, 128, step=8)

    # Initialize Model
    input_dim = 784  # Example for an image dataset (28x28 flattened)
    output_dim = 10  # Number of classes (e.g., digits 0-9)
    model = NN(input_dim, output_dim, hidden_layer, num_neurons)

    learning_rate = 0.01
    epochs = 50

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate , weight_decay=1e-4)

    # Set model to training mode
    model.train()

    # Training Loop
    for epoch in range(epochs):
        total_epoch_loss = 0  # Initialize loss

        for batch_features, batch_labels in train_loader:
            optimizer.zero_grad()  # Reset gradients
            outputs = model(batch_features)  # Forward pass
            loss = criterion(outputs, batch_labels)  # Compute loss
            loss.backward()  # Backpropagation
            optimizer.step()  # Update weights

            total_epoch_loss += loss.item()  # Track loss

        average_loss = total_epoch_loss / len(train_loader)  # Compute average loss
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {average_loss:.4f}")

    # Set model to evaluation mode
    model.eval()

    # Evaluate model accuracy
    with torch.no_grad():
        correct = 0
        total = 0
        for batch_features, batch_labels in test_loader:
            outputs = model(batch_features)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()

        accuracy = 100 * correct / total  # Compute accuracy

    # Track accuracy in Optuna
    trial.set_user_attr("accuracy", accuracy)

    return accuracy


In [ ]:
import optuna

In [ ]:
study  = optuna.create_study(direction = "maximize")
study.optimize(objective , n_trials =  10)